# Build SHHS outcome CSVs (survival)

Generates the outcome tables used by `scripts/train.py` / `scripts/eval.py` from official NSRR files in `shhs_csv/`.

**Inputs** (download from [sleepdata.org](https://sleepdata.org/datasets/shhs/files/datasets)):

- `shhs-cvd-summary-dataset-0.21.0.csv`
- `shhs-cvd-events-dataset-0.21.0.csv` (SHHS2 time fix for events after visit 2)
- `shhs2-dataset-0.21.0.csv` (visit-2 table; column `stdatep` anchors follow-up times)
- `shhs_config_summary.json`, `shhs_config_events.json` (in repo)

**Outputs** (`shhs_csv/results/`):

- `shhs1_events_summary_from_summary_survival_not_filtered.csv`
- `shhs2_events_summary_from_summary_survival_not_filtered_events.csv`

`not_filtered` = no exclusion for baseline prevalent CVD (`prev_*`).

Missing event dates are left as NaN in the CSV; training uses `extract_csv` (`mask_events`, etc.) to set outcome/time to −1 when event=1 without a date.

SHHS2: summary outcomes; event times and `censdate` shifted to days since visit 2; negative post-visit times reconciled from `shhs-cvd-events-dataset`.

In [28]:
import json
import os
from pathlib import Path

%pip install numpy pandas
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts" / "train.py").exists():
    REPO_ROOT = REPO_ROOT.parent

SHHS_DIR = REPO_ROOT / "shhs_csv"
RESULTS_DIR = SHHS_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PATH_SUMMARY = SHHS_DIR / "shhs-cvd-summary-dataset-0.21.0.csv"
PATH_EVENTS = SHHS_DIR / "shhs-cvd-events-dataset-0.21.0.csv"
PATH_SHHS2_VISIT = SHHS_DIR / "shhs2-dataset-0.21.0.csv"
PATH_CONFIG_SUMMARY = SHHS_DIR / "shhs_config_summary.json"
PATH_CONFIG_EVENTS = SHHS_DIR / "shhs_config_events.json"

OUT_SHHS1 = RESULTS_DIR / "shhs1_events_summary_from_summary_survival_not_filtered.csv"
OUT_SHHS2 = RESULTS_DIR / "shhs2_events_summary_from_summary_survival_not_filtered_events.csv"

NSRR_TIME_RENAMES = {
    "chd_dthdt": "chd_death_date",
    "cvd_dthdt": "cvd_death_date",
    "stk_date": "stroke_date",
    "ang_date": "angina_date",
    "revpro_date": "revasc_proc_date",
}

for p in (PATH_SUMMARY, PATH_EVENTS, PATH_SHHS2_VISIT, PATH_CONFIG_SUMMARY, PATH_CONFIG_EVENTS):
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
with open(PATH_CONFIG_SUMMARY, encoding="utf-8") as f:
    config = json.load(f)
with open(PATH_CONFIG_EVENTS, encoding="utf-8") as f:
    events_config = json.load(f)

id_var = config["id_var"]
outcome_columns = list(config["outcome_time_mapping"].keys())
time_columns = list(config["outcome_time_mapping"].values())

df_summary = pd.read_csv(PATH_SUMMARY, low_memory=False)
df_summary.rename(columns=NSRR_TIME_RENAMES, inplace=True)

C:\Users\Calzoni\AppData\Local\Temp\ipykernel_33764\162556797.py:11: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_summary.rename(columns=NSRR_TIME_RENAMES, inplace=True)


## SHHS1 (baseline / visit 1)

In [30]:
# "not_filtered": keep participants with baseline prevalent CVD flags
summary_dataset1 = df_summary.loc[df_summary["censdate"] > 0].copy()

keep_cols = [id_var, "censdate"] + outcome_columns + time_columns
summary_dataset1 = summary_dataset1[keep_cols].copy()

summary_dataset1[outcome_columns] = summary_dataset1[outcome_columns].fillna(0)
summary_dataset1[outcome_columns] = (summary_dataset1[outcome_columns] > 0).astype(int)

# Do not impute missing event dates here; `extract_csv` (e.g. mask_events) handles event=1 with NaN time.

summary_dataset1.to_csv(OUT_SHHS1, index=False)
print(f"Wrote SHHS1: {OUT_SHHS1}  shape={summary_dataset1.shape}")

Wrote SHHS1: c:\Users\Calzoni\Desktop\ALE\DOTTORATO\ESTERO\MambaSleep-CVD\shhs_csv\results\shhs1_events_summary_from_summary_survival_not_filtered.csv  shape=(5787, 22)


## SHHS2 (follow-up from visit 2)

Merge `stdatep`, require follow-up after visit 2 (`censdate > stdatep`), shift event times **and** `censdate` relative to visit 2, then reconcile negative event times using the longitudinal **events** CSV.

In [32]:
df_shhs2_visit = pd.read_csv(PATH_SHHS2_VISIT, encoding="latin-1", low_memory=False)
visit_dates = df_shhs2_visit[[id_var, "stdatep"]].dropna()

summary_dataset2 = summary_dataset1.merge(visit_dates, on=id_var, how="inner")
summary_dataset2 = summary_dataset2.rename(columns={"stdatep": "SHHS2_date"})

summary_dataset2 = summary_dataset2.loc[
    summary_dataset2["censdate"] > summary_dataset2["SHHS2_date"]
].reset_index(drop=True)

for time_col in time_columns + ["censdate"]:
    summary_dataset2[time_col] = summary_dataset2[time_col] - summary_dataset2["SHHS2_date"]

summary_dataset2 = summary_dataset2.loc[summary_dataset2["censdate"] > 0].reset_index(drop=True)

In [33]:
events_dataset = pd.read_csv(PATH_EVENTS, low_memory=False)
event_to_num = {
    v.lower(): int(k) for k, v in events_config["outcome_variables_mapping"].items()
}

for col in outcome_columns:
    time_col = f"{col}_date"
    col_lower = col.lower()
    if col_lower not in event_to_num:
        continue
    event_num = event_to_num[col_lower]
    mask_neg = summary_dataset2[time_col] < 0
    for id_neg in summary_dataset2.loc[mask_neg, id_var].tolist():
        idx = summary_dataset2.index[summary_dataset2[id_var] == id_neg][0]
        shhs2_date = summary_dataset2.at[idx, "SHHS2_date"]
        mask_ev = (events_dataset[id_var] == id_neg) & (events_dataset["event"] == event_num)
        patient_times = events_dataset.loc[mask_ev, "event_dt"].values
        if len(patient_times) == 0:
            summary_dataset2.at[idx, col] = 0
            summary_dataset2.at[idx, time_col] = np.nan
            continue
        rel = patient_times - shhs2_date
        valid = rel[rel > 0]
        if len(valid) > 0:
            summary_dataset2.at[idx, col] = 1
            summary_dataset2.at[idx, time_col] = valid.min()
        else:
            summary_dataset2.at[idx, col] = 0
            summary_dataset2.at[idx, time_col] = np.nan

summary_dataset2 = summary_dataset2.drop(columns=["SHHS2_date"]).reset_index(drop=True)
summary_dataset2.to_csv(OUT_SHHS2, index=False)
print(f"Wrote SHHS2: {OUT_SHHS2}  shape={summary_dataset2.shape}")

Wrote SHHS2: c:\Users\Calzoni\Desktop\ALE\DOTTORATO\ESTERO\MambaSleep-CVD\shhs_csv\results\shhs2_events_summary_from_summary_survival_not_filtered_events.csv  shape=(2622, 22)
